# Jun-2026 LAPD ion-saturation current (Isat) — fluctuation explorer

Companion to `Jun2026_IV_explore.ipynb`.  That notebook walks the *swept*
Langmuir tips (complete I+V pairs) to extract Vp/Te/ne; this one reads a
**fixed-bias ion-saturation channel** and looks at how the Isat signal
*fluctuates* in time (raw trace + FFT).

Reader + FFT helpers live in `Jun2026_Isat.py` (which reuses the read path and
knobs from `Jun2026_IV.py`); this notebook just drives them.  **You pick the
Isat channel by hand** — the notebook prints every channel's description and the
run's probe settings, then you name the scope + channel.

**This stage saves nothing** — no files, no batch loop.

Workflow: configure → list channel descriptions + run setup → select channel →
positions → read raw signal + plot. *(Pause here — FFT / fluctuation analysis
comes next.)*

## 1. Configure — set the run file

In [1]:
import importlib
get_ipython().run_line_magic("matplotlib", "qt")
# get_ipython().run_line_magic("matplotlib", "inline")

import numpy as np
import matplotlib.pyplot as plt

import Jun2026_IV as jiv
import Jun2026_Isat as jis
importlib.reload(jiv)
importlib.reload(jis)   # re-run this cell after editing Jun2026_Isat.py / Jun2026_IV.py

from data_analysis.io import open_lapd

# >>> SET THIS to the run you want to inspect <<<
# (a run with fixed-bias Isat channels, e.g. the 30-/32- 'Isat, ...' runs)
ifn = r"D:\data\LAPD\jun2026-jia\22-He-800G-bias40V-LP-p29-line_2026-06-12.hdf5"

# Current scaling knobs live in Jun2026_IV.py and are reused here (RESISTOR,
# Aprobe).  For fluctuation work the *shape* of the spectrum matters, not the
# absolute scaling.  The IV pipeline's I_SIGN is irrelevant to Isat and is not
# applied.
print(f"RESISTOR = {jiv.RESISTOR}, Aprobe = {jiv.Aprobe}")

assert ifn, "Set `ifn` to a run file first."
run = open_lapd(ifn)
print("backend:", run.backend)

RESISTOR = 25.0, Aprobe = 0.002
backend: pydaq


## 2. List channel descriptions + run setup

Print every scope group's channel descriptions and the run's hand-written
description (plasma / bias / probe settings).  Read these to decide which scope +
channel is the Isat channel you want — the selection is made by hand in the next
cell, nothing is guessed here.

In [2]:
channels = jis.list_all_channels(ifn)
print()
jis.print_run_description(ifn);

Scope groups and channel descriptions:

  scope 'scope':
    C1: 'I@P33, L'
    C2: 'V@P33, L'
    C3: 'I@P33, R'
    C4: 'V@P33, R'

=== Run description ===
Investigation of instability with multi-electrode biasing plate.
Langmuir sweep that ends right before instability occurs, 

Operator: Jia Han
    
Setup:
    - Plasma condition: 
      - Heater 1860A,  bank 90V, anode-cathode 50V, current 4kA
      - Helium backside pressure 40 Psi, Puff voltage 75V for 30ms West+East
      - Discharge 20 ms, Pulsing 1/3 Hz, plasma breakdown ~12 ms
      - Pressure unknown
      - Port 20, density at 10 ms:  0.65e13 cm^-3
      - Port 29, density at 10 ms:  1e13 cm^-3

    - Magnetic field
      - Black magnets at south: 888 A (PS12, 13),
      - Yellow & Purple magnets: configured for uniform 800 G
      - Black magnets at north: 0 A (PS11)

    - Multi-electrode bias: Port 35 top, centered ~(0,0)
      - Five circular disc shaped electodes of diameter: 2", 4", 6", 8" and 10"
      - Bias pulser

## 3. Select the Isat channel

From the lists above, set the scope group and channel of the Isat signal you
want to inspect.

In [19]:
# >>> SET THESE to the Isat channel you want (from the lists above) <<<
scope_name = "machscope"
chan       = "C2"

desc = channels.get(scope_name, {}).get(chan, "(no description)")
print(f"Selected Isat channel: scope '{scope_name}' {chan}  ->  {desc!r}")
assert scope_name in channels, f"scope {scope_name!r} not in {list(channels)}"
assert chan in channels[scope_name], f"channel {chan!r} not in {list(channels[scope_name])}"

Selected Isat channel: scope 'machscope' C2  ->  'MP@P33, Isat-X-'


## 4. Positions — pick one to inspect

In [20]:
# If the run has more than one probe drive (multiple motion groups), set
# motion_group_name to the one whose probe carries the Isat channel above
# (read_lp_positions lists the groups in its error). None works for single-drive runs.
motion_group_name = None    # e.g. "<Hermes>    p29_LP"

pos_array, xpos, ypos, npos, nshot = jiv.read_lp_positions(ifn, motion_group_name)

pos_index = npos//2     # <<< which position to inspect (0 .. npos-1)
print(f"\nInspecting position index {pos_index} of {npos} "
      f"(x={xpos[pos_index] if pos_index < len(xpos) else '?'} cm), {nshot} shots")

Using motion group: '<Hermes>    p29_LP'
Positions: 61 unique (x: 61, y: 1), 10 shots/position, 610 total shots.

Inspecting position index 30 of 61 (x=0.0 cm), 10 shots


## 5. Read raw Isat signal at this position + plot

Per-shot Isat traces (scaled per the knobs) at the chosen position.  Inspect the
raw signal: the mean level is the ion-saturation current; the scatter about it
shot-to-shot and the wiggle within a shot are the fluctuations we'll FFT next.

In [21]:
# uses scope_name / chan selected in section 3
tarr, Iarr = jis.get_isat_at_position(run, scope_name, chan, npos, nshot, pos_index)
print("Iarr:", Iarr.shape, " tarr:", tarr.shape)

fig, ax = plt.subplots(figsize=(11, 4))
for s in range(Iarr.shape[0]):
    ax.plot(tarr * 1e3, Iarr[s], lw=0.6, alpha=0.5)
ax.plot(tarr * 1e3, Iarr.mean(0), "k", lw=1.2, label="shot mean")
ax.set_xlabel("t [ms]"); ax.set_ylabel("Isat [a.u.]")
ax.set_title(f"Raw Isat — pos {pos_index} (x={xpos[pos_index] if pos_index < len(xpos) else '?'} cm), "
             f"scope '{scope_name}' {chan}: {desc!r}")
ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

Iarr: (10, 2500001)  tarr: (2500001,)


## 6. Downsampling options — compare on 1 shot at 3 positions

The raw Isat trace is ~2.5M samples/shot, which makes plotting (and any
interactive figure carrying it) slow. Before the FFT stage, decide how to
**downsample** for display.

Three options, each reducing the sample count by the same factor `Q`:

1. **Stride** — `I[::Q]`: keep every Q-th sample. Cheapest, but it *aliases*:
   fluctuations above the new Nyquist fold back into the band. Fine for a quick
   look at the slow envelope, misleading for spectral content.
2. **Block mean** — average each non-overlapping block of `Q` samples. A crude
   boxcar low-pass + decimate: anti-aliases somewhat and is robust, so it tracks
   the *mean level / slow structure* well, but the boxcar has poor stopband
   rejection.
3. **`scipy.signal.decimate`** — polyphase FIR-filtered decimation: proper
   anti-aliasing filter then downsample. The principled choice when the
   downsampled trace will be FFT'd; preserves in-band fluctuations without
   aliasing.

Below: one shot (`shot_sel`) at 3 positions, raw trace overlaid with all three
options at factor `Q`. Read these to pick the option + factor for the FFT stage.

In [24]:
from scipy import signal

# --- downsampling factor + which single shot to show ---
Q        = 500   # decimation factor (same for all three options, so they're comparable)
shot_sel = 0     # which shot (0 .. nshot-1) at each position
TMIN_MS, TMAX_MS = 0.0, 10.0   # display window (ms) -- focus on 0..10 ms for now

# Three positions to compare (stationary probe, so these should look alike;
# differences are shot-to-shot / drift, not spatial). Clip to valid range.
pos_sel = [npos // 4, npos // 2, 3 * npos // 4]
pos_sel = sorted({min(max(p, 0), npos - 1) for p in pos_sel})


def ds_stride(t, x, q):
    """Option 1 -- keep every q-th sample (no filtering; aliases)."""
    return t[::q], x[::q]


def ds_blockmean(t, x, q):
    """Option 2 -- mean of each non-overlapping block of q samples (boxcar LPF)."""
    n = (x.size // q) * q
    xb = x[:n].reshape(-1, q).mean(axis=1)
    tb = t[:n].reshape(-1, q).mean(axis=1)
    return tb, xb


def ds_decimate(t, x, q):
    """Option 3 -- scipy polyphase FIR-filtered decimation (anti-aliased)."""
    # decimate caps q per call (FIR order grows with q); chain factors if large.
    xq = x
    qrem = q
    for step in _factor_chain(qrem):
        xq = signal.decimate(xq, step, ftype="fir", zero_phase=True)
    return t[::q][: xq.size], xq


def _factor_chain(q, cap=12):
    """Split a large decimation factor into steps each <= cap (decimate FIR limit)."""
    steps = []
    while q > cap:
        f = next((d for d in range(cap, 1, -1) if q % d == 0), cap)
        steps.append(f)
        q //= f
    if q > 1:
        steps.append(q)
    return steps


_options = [
    ("stride",        ds_stride,    "tab:orange"),
    ("block mean",    ds_blockmean, "tab:green"),
    ("decimate(FIR)", ds_decimate,  "tab:red"),
]

fig, axes = plt.subplots(len(pos_sel), 1, figsize=(12, 3.2 * len(pos_sel)),
                         sharex=True)
axes = np.atleast_1d(axes)

for ax, pidx in zip(axes, pos_sel):
    tarr_p, Iarr_p = jis.get_isat_at_position(run, scope_name, chan, npos, nshot, pidx)
    I = Iarr_p[shot_sel]
    t_ms = tarr_p * 1e3

    ax.plot(t_ms, I, color="0.6", lw=0.5, label=f"raw ({I.size:,} pts)")
    for name, fn, c in _options:
        td, xd = fn(tarr_p, I, Q)
        ax.plot(td * 1e3, xd, color=c, lw=1.0, alpha=0.9,
                label=f"{name} (Q={Q}, {xd.size:,} pts)")

    xcm = xpos[pidx] if pidx < len(xpos) else "?"
    ax.set_title(f"pos {pidx} (x={xcm} cm), shot {shot_sel}", fontsize=10)
    ax.set_ylabel("Isat [a.u.]")
    ax.set_xlim(TMIN_MS, TMAX_MS)   # focus on 0..10 ms
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", fontsize=8, ncol=2)

axes[-1].set_xlabel("t [ms]")
fig.suptitle(f"Isat downsampling options (factor Q={Q})  —  "
             f"scope '{scope_name}' {chan}: {desc!r}", fontsize=11)
plt.tight_layout()
plt.show()


---
**Pause here.** Raw Isat signal read and plotted for one position/channel.
Next step: FFT the fluctuations (per-shot + averaged spectra) — to be added.